# Pipeline

A **Pipeline** is a machine learning utility provided by **Scikit-learn** that allows multiple preprocessing steps and the machine learning model to be combined into a **single workflow**.

Instead of manually performing each preprocessing step and then training the model, a Pipeline executes every step automatically in the correct order.

A typical pipeline looks like this:

```text
Raw Data
    │
    ▼
Data Preprocessing
    │
    ▼
Feature Scaling
    │
    ▼
Model Training
    │
    ▼
Prediction
```

---

# Why do we need a Pipeline?

Without a Pipeline, we usually perform each preprocessing step manually.

For example:

1. Split the dataset.
2. Scale the training data.
3. Scale the test data.
4. Train the model.
5. Make predictions.

This works, but it becomes difficult to manage when there are multiple preprocessing steps.

A Pipeline combines all these steps into a single object, making the code cleaner, easier to read, and less error-prone.

---

# What is Data Leakage?

One of the biggest reasons for using a Pipeline is to prevent **Data Leakage**.

**Data Leakage** occurs when information from the **test data** or **validation data** accidentally influences the training process.

For example, suppose we standardize the entire dataset **before** splitting it into training and testing data.

```python
scaler.fit(X)
```

Here, the scaler calculates the **mean** and **standard deviation** using **all data**, including the test set.

As a result, the model indirectly gains information about the test data before making predictions.

This produces overly optimistic results because the model has already "seen" information from the test dataset.

> **Data Leakage = The model accidentally learns information from the test or validation data before evaluation.**

---

# How does Pipeline prevent Data Leakage?

When using a Pipeline, every preprocessing step is performed **inside each training fold** during Cross Validation.

For example, consider one fold of Cross Validation.

```text
Training Fold
      │
      ▼
Fit StandardScaler
      │
      ▼
Transform Training Data
      │
      ▼
Train kNN Model
      │
      ▼
Transform Validation Data
(using the same scaler)
      │
      ▼
Evaluate Model
```

Notice that the **StandardScaler is fitted only on the training data**.

The validation or test data is **never used** while calculating the mean and standard deviation.

Instead, the same scaler learned from the training data is simply applied to transform the validation or test data.

This completely prevents data leakage.

---

# Understanding the Pipeline Code

### Step 1: Create the Pipeline

```python
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])
```

The Pipeline contains two steps.

1. **StandardScaler()** scales the features.
2. **KNeighborsClassifier()** trains the machine learning model.

These steps are executed sequentially.

```text
Input Data
      │
      ▼
StandardScaler
      │
      ▼
kNN Classifier
      │
      ▼
Prediction
```

---

### Step 2: Define the Hyperparameter Grid

```python
param_grid = {
    "knn__n_neighbors": [3, 5, 7, 9]
}
```

Since the classifier is inside the Pipeline and is named **knn**, its hyperparameters are accessed using:

```text
step_name__parameter_name
```

Therefore,

```python
knn__n_neighbors
```

means:

> Change the **n_neighbors** parameter of the **knn** step inside the Pipeline.

GridSearchCV will test:

- k = 3
- k = 5
- k = 7
- k = 9

---

### Step 3: Apply GridSearchCV

```python
classifierCV = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="recall"
)
```

GridSearchCV now performs **5-Fold Cross Validation** on the entire Pipeline.

For every value of **k**, the following operations are performed automatically.

```text
Training Fold
      │
      ▼
Fit StandardScaler
      │
      ▼
Transform Training Data
      │
      ▼
Train kNN
      │
      ▼
Transform Validation Data
      │
      ▼
Evaluate Recall Score
```

This process repeats for every fold and every value of **k**.

---

### Step 4: Train the Pipeline

```python
classifierCV.fit(X_train, y_train)
```

Notice that **X_train is not manually scaled**.

The Pipeline automatically:

- Fits the StandardScaler.
- Scales the training data.
- Trains the kNN model.
- Performs Cross Validation.
- Selects the best value of **k**.

All of this happens automatically with a single line of code.

---

### Step 5: Make Predictions

```python
y_pred = classifierCV.predict(X_test)
```

Again, notice that **X_test is not manually scaled**.

The Pipeline automatically:

- Uses the scaler learned from the training data.
- Transforms the test data.
- Uses the best kNN model to make predictions.

No additional preprocessing code is required.

---

# Advantages of Pipeline

- Makes the code cleaner and easier to read.
- Executes preprocessing steps automatically.
- Prevents data leakage.
- Works seamlessly with Cross Validation and GridSearchCV.
- Reduces the chances of human error.
- Makes machine learning workflows easier to maintain.

---

# Summary

A **Pipeline** combines preprocessing steps and the machine learning model into a single workflow. During Cross Validation, every preprocessing step is fitted only on the training data and then applied to the validation or test data. This prevents **data leakage**, simplifies the code, and allows GridSearchCV to safely tune hyperparameters without requiring manual preprocessing.

In [1]:
# Pipeline with GridSearchCV for kNN

import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score

data=pd.read_csv("heart.csv")
X=data.drop("target", axis=1)
y=data["target"]

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# we do not need to manually scale the data now. pipeline will scale the data and train the model

# Create the Pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

# Hyperparameter values to test
param_grid = {
    "knn__n_neighbors": [3, 5, 7, 9]
}

# GridSearchCV with Pipeline
classifierCV = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="recall"
)

# Train the pipeline
classifierCV.fit(X_train, y_train)

# Make predictions
y_pred = classifierCV.predict(X_test)

# Evaluation
print("Recall Score    :", recall_score(y_test, y_pred))
print("Accuracy Score  :", accuracy_score(y_test, y_pred))
print("Precision Score :", precision_score(y_test, y_pred))

# Best Hyperparameter
print("\nBest Parameters:")
print(classifierCV.best_params_)

Recall Score    : 0.90625
Accuracy Score  : 0.9180327868852459
Precision Score : 0.9354838709677419

Best Parameters:
{'knn__n_neighbors': 7}


# Purpose of Important Machine Learning Tools

As machine learning projects become larger, manually handling preprocessing, model selection, and evaluation becomes difficult. Libraries such as **Scikit-learn** provide utilities like **Train-Test Split**, **Cross Validation**, **GridSearchCV**, and **Pipeline** to automate these tasks and build more reliable models.

---

# 1. Train-Test Split

### Purpose

The purpose of **Train-Test Split** is to divide the dataset into **training data** and **testing data**.

The training data is used to train the model, while the testing data is kept completely unseen and is used to evaluate the final performance of the trained model.

Without a train-test split, we cannot determine how well the model performs on new, unseen data.

---

# 2. Validation Set

### Purpose

The purpose of the **Validation Set** is to evaluate the model during training and help choose the best model or the best hyperparameters.

Instead of repeatedly testing different models on the test dataset, we use the validation dataset for model selection and hyperparameter tuning.

This keeps the test dataset completely unseen until the final evaluation.

---

# 3. Cross Validation

### Purpose

The purpose of **Cross Validation** is to produce a **more reliable estimate** of model performance.

Instead of evaluating the model using a single validation split, Cross Validation trains and validates the model multiple times using different portions of the dataset.

This reduces the effect of random train-validation splits and makes better use of the available data.

---

# 4. K-Fold Cross Validation

### Purpose

The purpose of **K-Fold Cross Validation** is to ensure that **every data sample gets a chance to be used as validation data**.

The training dataset is divided into **K equal folds**. During each iteration, one fold acts as the validation set while the remaining folds are used for training.

The final performance is calculated by averaging the scores from all iterations.

This provides a more stable and trustworthy estimate of model performance.

---

# 5. GridSearchCV

### Purpose

The purpose of **GridSearchCV** is to **automatically find the best hyperparameter values** for a machine learning model.

Instead of manually testing different hyperparameters, GridSearchCV tries every possible combination, evaluates each one using **Cross Validation**, and selects the combination with the highest performance score.

For example, in kNN it automatically determines the best value of **k**.

GridSearchCV also retrains the final model using the best hyperparameters, so no additional training is required.

> **Purpose:** Automatically perform hyperparameter tuning and select the best model.

---

# 6. Pipeline

### Purpose

The purpose of a **Pipeline** is to combine all preprocessing steps and the machine learning model into **one automated workflow**.

Instead of manually scaling data, training the model, and making predictions separately, the Pipeline performs every step in the correct order automatically.

Pipelines also prevent **Data Leakage** because preprocessing operations are fitted only on the training data and then applied to the validation or test data.

> **Purpose:** Automate preprocessing and model training while preventing data leakage.

---

# 7. StandardScaler

### Purpose

The purpose of **StandardScaler** is to standardize numerical features so that they have a similar scale.

It transforms every feature to have:

- Mean = 0
- Standard Deviation = 1

This prevents features with larger numerical values from dominating distance calculations.

StandardScaler is especially important for distance-based algorithms such as **kNN**.

---

# 8. Hyperparameters

### Purpose

Hyperparameters are values that are chosen **before training begins**.

They control how the machine learning algorithm learns from the data but are **not learned automatically** by the model.

Examples include:

- Number of neighbors (**k**) in kNN
- Learning rate
- Maximum tree depth
- Number of decision trees

The purpose of hyperparameter tuning is to find the values that produce the best model performance.

---

# 9. Data Leakage

### Purpose

Data Leakage is **not a tool**, but an important concept in machine learning.

It occurs when information from the validation or test dataset accidentally becomes available during training.

This results in overly optimistic evaluation scores because the model has indirectly seen the unseen data.

Pipelines are commonly used to prevent data leakage.

---

# Summary

| Tool / Concept | Purpose |
|----------------|---------|
| **Train-Test Split** | Split data into training and testing datasets. |
| **Validation Set** | Tune hyperparameters and select the best model. |
| **Cross Validation** | Obtain a more reliable estimate of model performance. |
| **K-Fold Cross Validation** | Use every sample as validation data once and average the results. |
| **GridSearchCV** | Automatically find the best hyperparameters using Cross Validation. |
| **Pipeline** | Automate preprocessing and model training while preventing data leakage. |
| **StandardScaler** | Scale features to a common range for better model performance. |
| **Hyperparameters** | Control the behavior of the learning algorithm before training. |
| **Data Leakage** | Prevent the model from learning information from unseen data. |